# 🎙️ Vocalyx — Google Colab Dedicated GPU Server

Run the entire **Vocalyx** AI Voice Telephony & Cloned Voice Platform on a free **NVIDIA T4 GPU (16 GB VRAM)** on Google Colab.

### Why Colab?
- ⚡ **Dedicated NVIDIA T4 GPU:** Instant sub-second TTS synthesis (~0.8s – 1.2s).
- 🚀 **Zero Queue Latency:** Model stays permanently loaded in VRAM—no timeouts or dropped calls.
- 🌐 **Public HTTPS URL:** Live Ngrok tunnel for Twilio webhooks.

## Step 1: Ensure T4 GPU is Enabled
Click **Runtime** > **Change runtime type** > Select **T4 GPU** (Hardware accelerator).

In [ ]:
# Verify GPU
!nvidia-smi

## Step 2: Install Dependencies & Apply Float32 Patches

In [ ]:
# 1. Install Chatterbox Turbo TTS and runtime dependencies
!pip install -q chatterbox-tts --no-deps
!pip install -q fastapi uvicorn[standard] pyngrok twilio groq sqlalchemy sentence-transformers faiss-cpu pypdf python-multipart soundfile diffusers conformer einops encodec s3tokenizer resemble-perth pykakasi pyloudnorm spacy-pkuseg

# 2. Apply Float32 / NumPy 2.x compatibility patches
!find /usr/local/lib -name "s3tokenizer.py" -exec sed -i 's/self\._mel_filters\.to(self\.device) @ magnitudes/self._mel_filters.float().to(self.device) @ magnitudes.float()/g' {} +
!find /usr/local/lib -name "voice_encoder.py" -exec sed -i 's/self\.lstm(mels)/self.lstm(mels.float())/g' {} +
!find /usr/local/lib -name "voice_encoder.py" -exec sed -i 's/self\.inference(mels\.to(self\.device)/self.inference(mels.float().to(self.device)/g' {} +

print("✅ Dependencies installed and Float32 patches applied successfully!")

## Step 3: Extract Vocalyx Code
Upload `vocalyx.zip` to the Colab files panel on the left (or run this cell after uploading).

In [ ]:
import os

!unzip -q -o /content/vocalyx.zip -d /content/vocalyx
%cd /content/vocalyx
!mkdir -p /content/vocalyx/backend/tts_output /content/vocalyx/backend/uploaded_docs

print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

## Step 4: Configure API Keys & Twilio Credentials

In [ ]:
import os

os.environ["USE_CHATTERBOX_TURBO"] = "true"
os.environ["CUDA_DEVICE"] = "cuda"
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")
os.environ["GROQ_MODEL"] = "openai/gpt-oss-20b"
os.environ["TELEGRAM_BOT_TOKEN"] = os.getenv("TELEGRAM_BOT_TOKEN", "")
os.environ["TELEGRAM_CHAT_ID"] = os.getenv("TELEGRAM_CHAT_ID", "")

# Twilio Credentials
os.environ["TWILIO_ACCOUNT_SID"] = os.getenv("TWILIO_ACCOUNT_SID", "")
os.environ["TWILIO_AUTH_TOKEN"] = os.getenv("TWILIO_AUTH_TOKEN", "")  # Enter your active Twilio Auth Token here
os.environ["TWILIO_PHONE_NUMBER"] = os.getenv("TWILIO_PHONE_NUMBER", "")
os.environ["TWILIO_VOICE_ENGINE"] = "cloned"

print("✅ Credentials configured!")

## Step 5: Pre-Warm Cloned Voice on Dedicated GPU
Caches the cloned voice conditionals in VRAM for instant ~1.0s synthesis.

In [ ]:
import sys
if "/content/vocalyx" not in sys.path:
    sys.path.insert(0, "/content/vocalyx")

from backend import tts_engine
print("🔥 Pre-warming voice clone on NVIDIA T4 GPU...")
res = tts_engine.prewarm_engine()
print(f"✅ Cloned voice ready at: {res}")

## Step 6: Start Ngrok & Run Live Vocalyx Server

In [ ]:
import os, subprocess, time
from pyngrok import ngrok, conf

# 1. Clean up old processes
os.system("pkill -9 -f 'uvicorn' 2>/dev/null")
try:
    ngrok.kill()
except Exception:
    pass

# 2. Connect Ngrok tunnel on port 8000
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "")
conf.get_default().auth_token = NGROK_AUTH_TOKEN
try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
except Exception as _ngrok_err:
    if "already set" not in str(_ngrok_err).lower():
        raise
tunnel = ngrok.connect(8000)
public_url = str(tunnel.public_url).rstrip("/")
os.environ["PUBLIC_BASE_URL"] = public_url

print("=" * 65)
print(f"🚀 VOCALYX BACKEND IS LIVE (DEDICATED T4 GPU)!")
print(f"🌐 PUBLIC BASE URL: {public_url}")
print(f"📞 TWILIO WEBHOOK URL: {public_url}/api/twilio/voice")
print("=" * 65)
print("👉 Paste this URL into your Netlify App Settings as 'Public Server URL'.")
print("\n--- SERVER LOGS ---\n")

# 3. Run FastAPI via Uvicorn with the public URL injected
proc = subprocess.Popen(
    ["python", "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=dict(os.environ, PUBLIC_BASE_URL=public_url, PORT="8000")
)
time.sleep(2)

try:
    for line in proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\nShutting down server...")
    proc.terminate()
    ngrok.kill()